In [1]:
# Uninstall original Albumentations if installed
!uv pip uninstall albumentations

Using Python 3.12.12 environment at: /usr
Uninstalled 1 package in 171ms
 - albumentations==2.0.8


In [2]:
!uv pip install -q torchmetrics albumentationsx

In [3]:
import os
import random
import time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Callable, Dict, Tuple
import logging

import numpy as np
import pandas as pd
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
from torch import nn, optim
import torch.backends.cudnn as cudnn

# from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet34, ResNet34_Weights
from torchmetrics import Accuracy
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from torch.optim import AdamW

import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from tqdm.notebook import tqdm  # Better progress bars for notebooks

In [4]:
# mount drive
import os
from google.colab import drive
drive.mount('/content/drive')

# Change the current working directory
new_dir = "/content/drive/MyDrive/"
os.chdir(new_dir)

# Get the current working directory
current_dir = os.getcwd()
print(f"Current working directory after changing: {current_dir}")

Mounted at /content/drive
Current working directory after changing: /content/drive/MyDrive


In [5]:
# ----------------------------
# Hardware Check
# ----------------------------

def check_hardware() -> torch.device:
    if not torch.cuda.is_available():
        logger.warning("No GPU detected. Using CPU.")
        return torch.device("cpu")
    device_index = torch.cuda.current_device()
    device = torch.device(f"cuda:{device_index}")
    name = torch.cuda.get_device_name()
    mem = torch.cuda.get_device_properties().total_memory / 1024**3
    print(f"Training on GPU: {name} with {mem:.1f} GB")
    return device

import multiprocessing
# sklearn and multiprocessing
cores = multiprocessing.cpu_count()
print(f"Number of CPU cores: {cores}")
check_hardware()

Number of CPU cores: 8
Training on GPU: Tesla T4 with 14.7 GB


device(type='cuda', index=0)

In [6]:
#@title resnet34 architecture
model = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
print(model)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 212MB/s]

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Training logic

- Phase 1: unfreeze batchNorm layers and the classifier
- Phase 2: unfreeze batchNorm layers, layer3, layer4, the classifier

In [7]:
#@title RESNET34

# ----------------------------
# Logger Setup
# ----------------------------
LOGGER_NAME = "RESNET34"
logger = logging.getLogger(LOGGER_NAME)

def setup_logging(log_level: str = "INFO") -> None:
    """Sets up a console logger."""
    logger.setLevel(getattr(logging, log_level.upper(), logging.INFO))
    if logger.hasHandlers():
        for h in list(logger.handlers):
            logger.removeHandler(h)
    ch = logging.StreamHandler()
    ch.setLevel(logger.level)
    ch.setFormatter(logging.Formatter("%(asctime)s %(levelname)s: %(message)s"))
    logger.addHandler(ch)
    logger.propagate = False

setup_logging()

# ----------------------------
# Reproducibility
# ----------------------------
def set_seed(seed: int) -> None:
    """Sets random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        cudnn.deterministic = False
        cudnn.benchmark = True

# ----------------------------
# Hardware Check
# ----------------------------
def check_hardware() -> torch.device:
    """Checks for available hardware (GPU/CPU) and reports it."""
    if not torch.cuda.is_available():
        logger.warning("No GPU detected. Using CPU.")
        return torch.device("cpu")
    device_index = torch.cuda.current_device()
    device = torch.device(f"cuda:{device_index}")
    name = torch.cuda.get_device_name()
    mem = torch.cuda.get_device_properties().total_memory / 1024**3
    logger.info(f"Training on GPU: {name} with {mem:.1f} GB")
    return device

# ----------------------------
# Configuration
# ----------------------------
@dataclass
class Config:
    model_name: str = "resnet34"
    base_epochs: int = 3
    finetune_epochs: int = 20
    batch_size: int = 32
    num_workers: int = 8
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    dropout_rate: float = 0.5
    scheduler_factor: float = 0.1
    scheduler_patience: int = 5
    early_stopping_patience: int = 5
    seed: int = 123

    data_dir: Path = Path("./batdrive/OC/P6/P6_data/Images/")
    data_csv: Path = Path("./batdrive/OC/P6/P6_data/df_cleaned.csv")
    output_dir: Path = Path("./batdrive/models/outputs/resnet")

    save_all_misclassified: bool = True
    required_columns: list = field(default_factory=lambda: ["image", "level_1"])

    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------
# Data Loading & Validation
# ----------------------------
def validate_csv_columns(csv_path: Path, required_columns: list) -> None:
    """Checks if the required columns exist in the CSV file."""
    df_cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
    missing = set(required_columns) - set(df_cols)
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}")


def load_dataframe(csv_path: Path, required_columns: list) -> pd.DataFrame:
    """Loads and validates the main dataframe."""
    validate_csv_columns(csv_path, required_columns)
    return pd.read_csv(csv_path, usecols=required_columns)


class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: A.Compose, data_dir: Path):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.data_dir = data_dir

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        img_filename = self.df.loc[idx, "image"]
        img_path = self.data_dir / img_filename
        label = int(self.df.loc[idx, "label_encoded"])

        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f"Image at path {img_path} could not be loaded.")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transform(image=image)["image"]
        return image, label, img_filename


def get_transforms(is_training: bool) -> A.Compose:
    """Returns the augmentation pipeline for training or validation."""
    if is_training:
        return A.Compose([
            A.SmallestMaxSize(max_size=256, interpolation=cv2.INTER_AREA),
            A.Affine(
                scale=(0.96, 1.04),
                translate_percent=0.2,
                rotate=(-15, 15),
                shear=0,
                border_mode=cv2.BORDER_REPLICATE,
                p=0.5
            ),
            A.RandomCrop(height=224, width=224),
            A.HorizontalFlip(),
            A.OneOf([
                A.RandomBrightnessContrast(),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
            ], p=1.0),
            A.OneOf([A.GaussNoise(p=0.3), A.GaussianBlur(p=0.3)], p=0.3),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.SmallestMaxSize(max_size=256, interpolation=cv2.INTER_AREA),
            A.CenterCrop(height=224, width=224),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

# ----------------------------
# Model & Utilities
# ----------------------------
class EarlyStopping:
    """Stops training when a monitored metric has stopped improving."""
    def __init__(self, patience: int, delta: float, save_path: Path):
        self.patience = patience
        self.delta = delta
        self.save_path = save_path
        self.best_loss = float("inf")
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss: float, model: nn.Module) -> None:
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), str(self.save_path))
        else:
            self.counter += 1
            logger.info(f"EarlyStopping counter {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

def setup_model(num_classes: int, dropout_rate: float) -> nn.Module:
    """Sets up the ResNet34 model with a custom classifier head."""
    model = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)

    for param in model.parameters():
        param.requires_grad = False
    in_f = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(dropout_rate), nn.Linear(in_f, num_classes))
    return model


def update_trainable_params(model: nn.Module, phase: int) -> None:
    """Updates which model parameters are trainable based on the training phase."""
    for p in model.parameters():
        p.requires_grad = False
    if phase == 1: # Unfreeze classifier head and batch norm layers
        for p in model.fc.parameters():
            p.requires_grad = True
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d):
                for p in m.parameters():
                    p.requires_grad = True
    elif phase == 2: # Unfreeze deeper layers as well
        for layer in (model.layer3, model.layer4, model.fc):
            for p in layer.parameters():
                p.requires_grad = True
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d):
                for p in m.parameters():
                    p.requires_grad = True
    else:
        raise ValueError("Phase must be 1 or 2.")


def validate(model, loader, criterion, device, acc_metric: Accuracy, max_mis=float('inf')):
    """Performs a validation loop."""
    model.eval()
    loss_total = 0.0
    mis = {"filenames": [], "true": [], "pred": []}
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels, filenames in tqdm(loader, desc="Validating"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss_total += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            acc_metric.update(preds, labels)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

            diff_idxs = (preds != labels).nonzero(as_tuple=True)[0]
            for i in diff_idxs.cpu().tolist():
                if len(mis["filenames"]) >= max_mis:
                    break
                mis["filenames"].append(filenames[i])
                mis["true"].append(labels[i].item())
                mis["pred"].append(preds[i].item())

    total = len(loader.dataset)
    val_loss = loss_total / total
    val_acc = acc_metric.compute().item()
    acc_metric.reset()

    logger.info(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    return {
        "val_loss": val_loss,
        "val_acc": val_acc,
        "misclassified": mis,
        "all_preds": all_preds,
        "all_labels": all_labels,
    }

def train_epoch(model, loader, criterion, optimizer, device, acc_metric: Accuracy):
    """Performs a training epoch."""
    model.train()
    loss_total = 0.0

    for inputs, labels, _ in tqdm(loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        loss_total += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        acc_metric.update(preds, labels)

    total = len(loader.dataset)
    train_loss = loss_total / total
    train_acc = acc_metric.compute().item()
    acc_metric.reset()

    return train_loss, train_acc


# ----------------------------
# Plotting
# ----------------------------

def plot_learning_curves(train_m: Dict, val_m: Dict, out: Path) -> None:
    """Plots and saves learning curves for loss and accuracy."""
    epochs = range(1, len(train_m["loss"]) + 1)
    plt.figure()
    plt.plot(epochs, train_m["loss"], label="Train Loss")
    plt.plot(epochs, val_m["loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Loss vs. Epochs")
    plt.savefig(out / "loss_curve.png", dpi=300)
    plt.close()

    plt.figure()
    plt.plot(epochs, train_m["acc"], label="Train Acc")
    plt.plot(epochs, val_m["acc"], label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.title("Accuracy vs. Epochs")
    plt.savefig(out / "acc_curve.png", dpi=300)
    plt.close()

def plot_confusion_matrix(all_labels: List[int], all_preds: List[int], class_names: List[str], output_dir: Path) -> None:
    """
    Computes, plots, saves, and displays a confusion matrix.
    """
    logger = logging.getLogger(LOGGER_NAME)
    # 1) Compute matrix
    cm = confusion_matrix(all_labels, all_preds)
    n = len(class_names)

    # 2) Create figure & heatmap
    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    fig.colorbar(im, ax=ax)

    # 3) Tick marks and labels
    ax.set_xticks(np.arange(n))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticks(np.arange(n))
    ax.set_yticklabels(class_names)

    # 4) Titles
    ax.set_title('Confusion Matrix')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

    # 5) Annotate each cell with the integer value
    thresh = cm.max() / 2.0
    for i in range(n):
        for j in range(n):
            color = 'white' if cm[i, j] > thresh else 'black'
            ax.text(j, i, f"{cm[i, j]:d}",
                    ha='center', va='center', color=color)

    # 6) Layout, save, display, and close
    fig.tight_layout()
    save_path = output_dir / 'confusion_matrix.png'
    fig.savefig(save_path, dpi=300)
    logger.info(f"Confusion matrix saved to {save_path}")
    plt.close(fig)

def plot_misclassified(mis: Dict[str, List],
                       class_names: List[str],
                       data_dir: Path,
                       output_dir: Path,
                       save_all: bool) -> None:
    """
    Plots misclassified samples in a grid.
    """
    n_total = len(mis['filenames'])
    n = n_total if save_all else min(16, n_total)
    if n == 0:
        logger.info("No misclassified samples to plot.")
        return

    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig = plt.figure(figsize=(cols * 4, rows * 4.5))
    fig.suptitle("Misclassified Samples", fontsize=16)

    for idx in range(n):
        ax = fig.add_subplot(rows, cols, idx + 1)

        img_path = data_dir / mis['filenames'][idx]
        img = cv2.imread(str(img_path))
        if img is None:
            ax.text(0.5, 0.5, "Image not found", ha="center", va="center")
            ax.axis("off")
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.axis("off")

        true_lbl = class_names[mis['true'][idx]]
        pred_lbl = class_names[mis['pred'][idx]]
        title = f"True: {true_lbl}\nPred: {pred_lbl}"
        ax.set_title(title, fontsize=10)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(output_dir / "misclassified_samples.png", dpi=300)
    plt.close(fig)

# ----------------------------
# Training Pipeline
# ----------------------------

def train_model(config: Config):
    """Main function to run the entire training and evaluation pipeline."""
    set_seed(config.seed)

    df = load_dataframe(config.data_csv, config.required_columns)
    le = LabelEncoder()
    df["label_encoded"] = le.fit_transform(df["level_1"])
    class_names = le.classes_.tolist()
    num_classes = len(class_names)

    train_df, val_df = train_test_split(
        df, test_size=0.2, stratify=df["label_encoded"], random_state=config.seed
    )

    train_ds = ImageDataset(train_df, get_transforms(True), config.data_dir)
    val_ds   = ImageDataset(val_df,   get_transforms(False), config.data_dir)

    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers, pin_memory=True)

    device = check_hardware()
    model  = setup_model(num_classes, config.dropout_rate).to(device)
    criterion = nn.CrossEntropyLoss()

    train_acc_metric = Accuracy(task="multiclass", num_classes=num_classes).to(device)
    val_acc_metric   = Accuracy(task="multiclass", num_classes=num_classes).to(device)

    train_metrics = {"loss": [], "acc": []}
    val_metrics   = {"loss": [], "acc": []}

    # === Phase 1: Training classifier head ===
    logger.info("=== Phase 1: Training classifier head ===")
    ckpt1 = config.output_dir / "best_phase1.pth"
    es1   = EarlyStopping(config.early_stopping_patience, delta=1e-4, save_path=ckpt1)
    update_trainable_params(model, phase=1)
    opt1 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=config.learning_rate, weight_decay=config.weight_decay)
    sched1 = optim.lr_scheduler.ReduceLROnPlateau(opt1, factor=config.scheduler_factor, patience=config.scheduler_patience)

    for epoch in range(config.base_epochs):
        logger.info(f"Phase 1 — Epoch {epoch+1}/{config.base_epochs}")
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, opt1, device, train_acc_metric)
        val_res = validate(model, val_loader, criterion, device, val_acc_metric)

        sched1.step(val_res["val_loss"])
        es1(val_res["val_loss"], model)

        train_metrics["loss"].append(tr_loss); train_metrics["acc"].append(tr_acc)
        val_metrics["loss"].append(val_res["val_loss"]); val_metrics["acc"].append(val_res["val_acc"])

        if es1.early_stop:
            logger.info("Early stopping triggered in Phase 1")
            break

    logger.info(f"Loading best model from Phase 1 ({ckpt1})")
    model.load_state_dict(torch.load(ckpt1, map_location=device, weights_only=True))

    # === Phase 2: Fine-tuning deeper layers ===
    logger.info("=== Phase 2: Fine-tuning deeper layers ===")
    ckpt2 = config.output_dir / "best_phase2.pth"
    es2   = EarlyStopping(config.early_stopping_patience, delta=1e-4, save_path=ckpt2)
    update_trainable_params(model, phase=2)
    opt2 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=config.learning_rate * 0.1, weight_decay=config.weight_decay)
    sched2 = optim.lr_scheduler.ReduceLROnPlateau(opt2, factor=config.scheduler_factor, patience=config.scheduler_patience)

    for epoch in range(config.finetune_epochs):
        logger.info(f"Phase 2 — Epoch {epoch+1}/{config.finetune_epochs}")
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, opt2, device, train_acc_metric)
        val_res = validate(model, val_loader, criterion, device, val_acc_metric)

        sched2.step(val_res["val_loss"])
        es2(val_res["val_loss"], model)

        train_metrics["loss"].append(tr_loss); train_metrics["acc"].append(tr_acc)
        val_metrics["loss"].append(val_res["val_loss"]); val_metrics["acc"].append(val_res["val_acc"])

        if es2.early_stop:
            logger.info("Early stopping triggered in Phase 2")
            break

    logger.info(f"Loading best model from Phase 2 ({ckpt2})")
    model.load_state_dict(torch.load(ckpt2, map_location=device, weights_only=True))

    # === Final Evaluation and Reporting ===
    logger.info("=== Final Evaluation ===")
    final_res = validate(model, val_loader, criterion, device, val_acc_metric)

    mis = final_res["misclassified"]
    if mis["filenames"]:
        df_mis = pd.DataFrame({
            "True class": [class_names[i] for i in mis["true"]],
            "Predicted class": [class_names[i] for i in mis["pred"]]
        }, index=mis["filenames"])
        df_mis.index.name = "Image File"
        csv_path = config.output_dir / "misclassified_samples.csv"
        df_mis.to_csv(csv_path)
        logger.info(f"Saved misclassified DataFrame to {csv_path}")

    report = classification_report(final_res["all_labels"], final_res["all_preds"], target_names=class_names)
    with open(config.output_dir / "classification_report.txt", "w") as f:
        f.write(report)

    # === Plotting Results ===
    plot_learning_curves(train_metrics, val_metrics, config.output_dir)
    plot_confusion_matrix(final_res["all_labels"], final_res["all_preds"], class_names, config.output_dir)
    if mis["filenames"]:
        plot_misclassified(mis, class_names, config.data_dir, config.output_dir, config.save_all_misclassified)

    return model

# ----------------------------
# Main Execution Block
# ----------------------------
if __name__ == "__main__":
    try:
        cfg = Config(save_all_misclassified=True)
        resnet_model = train_model(cfg)
    except Exception as e:
        logger.exception(f"An error occurred during training: {e}")

2025-11-21 18:12:18,108 INFO: Training on GPU: Tesla T4 with 14.7 GB
2025-11-21 18:12:18,672 INFO: === Phase 1: Training classifier head ===
2025-11-21 18:12:18,674 INFO: Phase 1 — Epoch 1/3


Training:   0%|          | 0/27 [00:00<?, ?it/s]

Validating:   0%|          | 0/7 [00:00<?, ?it/s]

2025-11-21 18:15:47,362 INFO: Val Loss: 1.9423, Val Acc: 0.2143
2025-11-21 18:15:51,423 INFO: Phase 1 — Epoch 2/3


Training:   0%|          | 0/27 [00:00<?, ?it/s]

...

Training:   0%|          | 0/27 [00:00<?, ?it/s]

Validating:   0%|          | 0/7 [00:00<?, ?it/s]

2025-11-21 18:19:50,676 INFO: Val Loss: 0.5993, Val Acc: 0.8000
2025-11-21 18:19:50,677 INFO: EarlyStopping counter 1/5
2025-11-21 18:19:50,677 INFO: Loading best model from Phase 2 (batdrive/models/outputs/resnet/best_phase2.pth)
2025-11-21 18:19:50,845 INFO: === Final Evaluation ===


Validating:   0%|          | 0/7 [00:00<?, ?it/s]

2025-11-21 18:19:54,407 INFO: Val Loss: 0.5975, Val Acc: 0.8000
2025-11-21 18:19:54,989 INFO: Saved misclassified DataFrame to batdrive/models/outputs/resnet/misclassified_samples.csv
2025-11-21 18:19:59,081 INFO: Confusion matrix saved to batdrive/models/outputs/resnet/confusion_matrix.png


In [8]:
#@title Résultats

# 1. Définition du chemin
output_dir = Path("./batdrive/models/outputs/resnet")

print(f"Lecture des résultats depuis : {output_dir.resolve()}\n")

# ---------------------------------------------------------
# 2. Affichage du DataFrame des erreurs
# ---------------------------------------------------------
csv_path = output_dir / "misclassified_samples.csv"
if csv_path.exists():
    display(Markdown("### Misclassified samples"))
    df_mis = pd.read_csv(csv_path, index_col=0)
    display(df_mis)
else:
    print(f"Fichier non trouvé : {csv_path}")

# ---------------------------------------------------------
# 3. Affichage du rapport de classification
# ---------------------------------------------------------
report_path = output_dir / "classification_report.txt"
if report_path.exists():
    display(Markdown("### Classification report"))
    with open(report_path, "r") as f:
        print(f.read())
else:
    print(f"Fichier non trouvé : {report_path}")

# ---------------------------------------------------------
# 4. Affichage des graphiques
# ---------------------------------------------------------
image_files = [
    ("loss_curve.png", "Learning curves (Loss)"),
    ("acc_curve.png", "Learning curves (Accuracy)"),
    ("confusion_matrix.png", "Confusion matrix"),
    ("misclassified_samples.png", "Misclassified images")
]

for filename, title in image_files:
    img_path = output_dir / filename
    if img_path.exists():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(img_path)))
        print("\n")
    else:
        print(f"Image non trouvée : {filename}")

Output hidden; open in https://colab.research.google.com to view.

***